# 04 — Cycle Extraction & Genie Space

This notebook:

1. **Extracts cycle boundaries** from point-level predictions into a summary table (one row per detected cycle)
2. **Creates a Genie Space** — a natural language interface where engineers can ask questions about cycles without writing SQL

### What is a Genie Space?
Genie translates natural language questions into SQL, executes them, and presents results conversationally:

- *"How many brake cycles have been applied across all testbenches?"*
- *"What is the average brake cycle duration?"*
- *"Which testbench has the most cycles?"*
- *"What was the median pressure of each cycle since September?"*

In [ ]:
%run ./00_Config

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1. Extract Cycle Boundaries from Predictions

Convert point-level predictions (`True`/`False` per timestamp) into a **cycles summary table**: one row per detected cycle with timing and per-cycle statistics.

In [ ]:
pred_df = spark.table(PREDICTIONS_TABLE)
raw_df = spark.table(RAW_SENSOR_TABLE).select("test_run_id", "timestamp", "testbench_id")

pred_with_tb = pred_df.join(raw_df, on=["test_run_id", "timestamp"], how="left")
print(f"Predictions: {pred_with_tb.count():,} rows")

w = Window.partitionBy("test_run_id").orderBy("timestamp")

boundary_df = (
    pred_with_tb
    .withColumn("prev_prediction", F.lag("prediction", 1, 0).over(w))
    .withColumn("boundary",
                F.when(F.col("prediction") != F.col("prev_prediction"), 1).otherwise(0))
    .withColumn("cycle_group", F.sum("boundary").over(w))
)

cycle_segments = boundary_df.filter(F.col("prediction") == 1)

In [ ]:
cycles_df = (
    cycle_segments
    .groupBy("testbench_id", "test_run_id", "cycle_group")
    .agg(
        F.min("timestamp").alias("cycle_start"),
        F.max("timestamp").alias("cycle_end"),
        F.count("*").alias("n_points"),
        F.round(F.avg("pressure_bar"), 3).alias("mean_pressure_bar"),
        F.round(F.expr("percentile_approx(pressure_bar, 0.5)"), 3).alias("median_pressure_bar"),
        F.round(F.max("pressure_bar"), 3).alias("max_pressure_bar"),
        F.round(F.min("pressure_bar"), 3).alias("min_pressure_bar"),
        F.round(F.avg("temperature_c"), 2).alias("mean_temperature_c"),
        F.round(F.avg("flow_rate"), 3).alias("mean_flow_rate"),
        F.sum("valve_state").alias("valve_open_points"),
    )
    .withColumn("duration_seconds",
                F.round((F.unix_timestamp("cycle_end") - F.unix_timestamp("cycle_start")).cast("double")
                        + F.col("n_points") * 0.1, 1))
    .withColumn("cycle_id",
                F.concat(F.col("test_run_id"), F.lit("-C"), F.lpad(F.col("cycle_group").cast("string"), 3, "0")))
    .select(
        "cycle_id", "testbench_id", "test_run_id",
        "cycle_start", "cycle_end", "duration_seconds", "n_points",
        "mean_pressure_bar", "median_pressure_bar", "max_pressure_bar", "min_pressure_bar",
        "mean_temperature_c", "mean_flow_rate", "valve_open_points",
    )
    .orderBy("testbench_id", "cycle_start")
)

print(f"Detected {cycles_df.count()} cycles")
display(cycles_df.limit(20))

In [ ]:
cycles_df.write.mode("overwrite").saveAsTable(CYCLE_RESULTS_TABLE)
print(f"Cycles summary saved to {CYCLE_RESULTS_TABLE}")

### Quick Validation: Detected vs Ground Truth

In [ ]:
raw_df = spark.table(RAW_SENSOR_TABLE)
w_raw = Window.partitionBy("test_run_id").orderBy("timestamp")

gt_cycles = (
    raw_df
    .withColumn("prev_cycle_status", F.lag("cycle_status", 1, 0).over(w_raw))
    .filter((F.col("cycle_status") == 1) & (F.col("prev_cycle_status") == 0))
    .count()
)

detected_count = spark.table(CYCLE_RESULTS_TABLE).count()

print(f"Ground truth cycles: {gt_cycles}")
print(f"Detected cycles:     {detected_count}")
print(f"Detection rate:      {detected_count / gt_cycles * 100:.1f}%")

## 2. Add Table Comments for Genie

Genie uses table and column comments to understand the data. Well-documented tables produce better natural language -> SQL translations.

In [ ]:
spark.sql(f"""
    COMMENT ON TABLE {CYCLE_RESULTS_TABLE} IS 
    'Detected brake cycles from testbench sensor data. One row per cycle with timing, pressure, temperature, and flow statistics. Generated by the cycle detection ML pipeline.'
""")

column_comments = {
    "cycle_id": "Unique cycle identifier (format: testbench-run-cycle_number)",
    "testbench_id": "Brake testbench identifier (e.g. TB-PCU-001)",
    "test_run_id": "Test run identifier",
    "cycle_start": "Timestamp when the brake cycle started",
    "cycle_end": "Timestamp when the brake cycle ended",
    "duration_seconds": "Duration of the brake cycle in seconds",
    "n_points": "Number of sensor data points in this cycle",
    "mean_pressure_bar": "Average brake pressure during the cycle (bar)",
    "median_pressure_bar": "Median brake pressure during the cycle (bar)",
    "max_pressure_bar": "Maximum brake pressure during the cycle (bar)",
    "min_pressure_bar": "Minimum brake pressure during the cycle (bar)",
    "mean_temperature_c": "Average component temperature during the cycle (Celsius)",
    "mean_flow_rate": "Average hydraulic flow rate during the cycle (l/min)",
    "valve_open_points": "Number of data points where the magnet valve was open",
}

for col_name, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {CYCLE_RESULTS_TABLE} ALTER COLUMN {col_name} COMMENT '{comment}'")

print("Cycle results table: comments added.")

In [ ]:
spark.sql(f"""
    COMMENT ON TABLE {RAW_SENSOR_TABLE} IS 
    'Raw brake testbench sensor data sampled at 100ms intervals. Contains pressure, valve state, temperature, and flow rate readings for the Pressure Control Unit (PCU).'
""")

raw_comments = {
    "timestamp": "Measurement timestamp (100ms resolution)",
    "testbench_id": "Brake testbench identifier",
    "test_run_id": "Unique test run identifier",
    "pressure_bar": "Brake pressure in bar",
    "valve_state": "Magnet valve state (0=closed, 1=open)",
    "temperature_c": "Component temperature in degrees Celsius",
    "flow_rate": "Hydraulic flow rate in liters per minute",
    "cycle_status": "Ground truth label - 1 if this data point is part of a brake cycle, 0 if idle",
}

for col_name, comment in raw_comments.items():
    spark.sql(f"ALTER TABLE {RAW_SENSOR_TABLE} ALTER COLUMN {col_name} COMMENT '{comment}'")

print("Raw sensor table: comments added.")

## 3. Create the Genie Space

In [ ]:
import requests, json

host = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

warehouse_id = spark.conf.get("spark.databricks.clusterUsageTags.sqlWarehouseId", None)
if not warehouse_id:
    wh_resp = requests.get(f"https://{host}/api/2.0/sql/warehouses", headers=headers).json()
    warehouse_id = next(
        (w["id"] for w in wh_resp.get("warehouses", []) if w.get("state") == "RUNNING"),
        wh_resp["warehouses"][0]["id"] if wh_resp.get("warehouses") else None,
    )
    print(f"Using SQL warehouse: {warehouse_id}")

GENIE_SPACE_NAME = "Brake Cycle Explorer"
GENIE_DESCRIPTION = (
    "Explore brake testbench data and cycle detection results for the Pressure Control Unit. "
    "Ask questions about cycle counts, durations, pressure statistics, and testbench performance. "
    "A 'cycle' is a segment of time-series data where the brake component is performing a braking operation."
)

SAMPLE_QUESTIONS = [
    "How many brake cycles have been detected across all testbenches?",
    "What is the average brake cycle duration in seconds?",
    "Which testbench has the most cycles?",
    "What was the median pressure of each cycle since September?",
    "Show me the 10 longest cycles by duration.",
    "What is the average max pressure per testbench?",
    "How many cycles had a max pressure above 9 bar?",
    "What is the total number of cycles per test run?",
]

In [ ]:
import uuid

def _id():
    return uuid.uuid4().hex

serialized_space = json.dumps({
    "version": 2,
    "config": {
        "sample_questions": [
            {"id": _id(), "question": [q]} for q in SAMPLE_QUESTIONS
        ],
    },
    "data_sources": {
        "tables": [
            {"identifier": CYCLE_RESULTS_TABLE, "description": ["Detected brake cycles with timing and sensor statistics"]},
            {"identifier": RAW_SENSOR_TABLE, "description": ["Raw testbench sensor data at 100ms resolution"]},
        ]
    },
    "instructions": {
        "text_instructions": [
            {"id": _id(), "content": [GENIE_DESCRIPTION]}
        ]
    },
})

payload = {
    "warehouse_id": warehouse_id,
    "title": GENIE_SPACE_NAME,
    "description": GENIE_DESCRIPTION,
    "serialized_space": serialized_space,
}

print(f"Creating Genie Space: '{GENIE_SPACE_NAME}'...")
resp = requests.post(f"https://{host}/api/2.0/genie/spaces", headers=headers, json=payload)
resp.raise_for_status()
space = resp.json()

print(f"\nGenie Space ready!")
print(f"  ID: {space.get('space_id', 'N/A')}")
print(f"  Tables: {CYCLE_RESULTS_TABLE}, {RAW_SENSOR_TABLE}")
print(f"  Sample questions: {len(SAMPLE_QUESTIONS)}")
print(f"\nOpen it in the Databricks UI to start asking questions.")

## 4. Quick Test Query

In [ ]:
test_query = f"""
    SELECT 
        testbench_id,
        COUNT(*) AS total_cycles,
        ROUND(AVG(duration_seconds), 1) AS avg_duration_sec,
        ROUND(AVG(median_pressure_bar), 2) AS avg_median_pressure,
        ROUND(AVG(max_pressure_bar), 2) AS avg_max_pressure
    FROM {CYCLE_RESULTS_TABLE}
    GROUP BY testbench_id
    ORDER BY total_cycles DESC
"""

print("Sample: 'Show cycle stats per testbench'")
display(spark.sql(test_query))

## Summary — End-to-End Pipeline

| Notebook | Component | Databricks Feature |
|----------|-----------|--------------------|    
| 01 | Synthetic testbench data | Delta tables in Unity Catalog |
| 02 | Windowed time-series features | **Feature Engineering Client** |
| 03 | Training, registration & inference | **MLflow** + **Feature Store** + **Model Registry** |
| 04 | Cycle extraction & data explorer | **Genie Space** |
